# Virtual staining: expanded 3D reference baseline

Train an attributed Allen 3D U-Net on a new plate-separated development cohort.
This is an ongoing research experiment, **not a validated staining product**.

1. Choose **Runtime → Change runtime type → GPU**. A free GPU may not be available.
   This notebook does not buy compute or enable a paid plan.
2. Choose **Runtime → Run all**. Approve Drive mounting to retain checkpoints.
3. Inspect the metrics and **all six validation galleries** at the end.

The notebook downloads about **11.46 GB** of public microscopy data and prepares
full 3D volumes. The six reserved test fields are never downloaded or evaluated.
Download and preprocessing time is additional to training. Runtime disconnections
can interrupt a run; this trainer currently saves checkpoints but does **not**
implement continuation. Do not overwrite an existing run directory.

The code is pinned. The data plan is verified against its frozen SHA-256.
Model code is credited to the Allen Institute and retains its non-commercial
licence. Our contribution is the data/experiment integration and evaluation.

This run uses the full width-32/depth-4 architecture, reference weight
initialisation, 32×64×64 patches, batch 4 and 10,000 updates. Its batch, budget
and dataset differ from the publication. It is **not an exact paper reproduction**.


In [ ]:
import os, sys, subprocess, json, hashlib, urllib.request
from pathlib import Path

REPOSITORY = "https://github.com/kiranmanoj05/virtual-microscopy-image-staining.git"
CODE_REVISION = "fa683e6764de876ca65b3145620c2cc2e24c07f1"
PLAN_SHA256 = "e61c58d4e1832d4191a9ced0132a3b399a6102a9c6f2bf992ad5947ac1b20dcc"
PLAN_URL = "https://raw.githubusercontent.com/kiranmanoj05/virtual-microscopy-image-staining/feat/volumetric-reference-baseline/examples/reference-cohort-2026-11-01.json"
PROJECT = Path("/content/virtual-staining")
DATA = Path("/content/staining-data")
DATA.mkdir(exist_ok=True)
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
subprocess.run(["git", "checkout", CODE_REVISION], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[ml]"], cwd=PROJECT, check=True)
os.chdir(PROJECT)
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU selected. Change the Colab runtime to GPU before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
print("Pinned source:", CODE_REVISION)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
STEPS = 10_000
BATCH_SIZE = 4
RUN_NAME = "reference-expanded-10000-seed17"
OUTPUT = Path("/content/drive/MyDrive/virtual-staining") / RUN_NAME
if OUTPUT.exists():
    raise FileExistsError("This run already exists. Inspect it or choose a new RUN_NAME; do not overwrite it.")
print("Checkpoints and complete results will be saved to:", OUTPUT)


In [ ]:
from collections import Counter
from virtual_staining.context_data import _read_frozen

with urllib.request.urlopen(PLAN_URL, timeout=60) as response:
    payload = response.read(1_000_001)
if len(payload) > 1_000_000:
    raise ValueError("Unexpectedly large cohort manifest")
PLAN = DATA / "cohort.json"
PLAN.write_bytes(payload)
plan, fingerprint = _read_frozen(PLAN, "plan_sha256")
if fingerprint != PLAN_SHA256:
    raise ValueError("The downloaded cohort does not match this notebook's frozen plan")
if any(f["group_id"] in plan["excluded_prior_groups"] for f in plan["fields"]):
    raise ValueError("The new cohort overlaps an old plate")
counts = Counter(f["split"] for f in plan["fields"])
if counts != {"train": 24, "validation": 6, "test": 6}:
    raise ValueError("Cohort completeness mismatch")
print("Frozen cohort:", dict(counts))
print("Only train and validation images will be downloaded. Test pixels remain reserved.")


In [ ]:
import time
from virtual_staining.pilot import download_pilot
from virtual_staining.volume_data import prepare_volumes, load_volumes

RAW = DATA / "raw"
RECEIPTS = DATA / "original-downloads.json"
VOLUMES = DATA / "volumes"
# Retry transient public-server failures. Completed, verified downloads are reused;
# only incomplete files generated inside this notebook's raw directory are removed.
for attempt in range(3):
    try:
        downloaded = download_pilot(PLAN, RAW, development_only=True)
        break
    except Exception:
        if attempt == 2:
            raise
        for field in plan["fields"]:
            if field["split"] == "test":
                continue
            partial = RAW / ("allen_fov_" + field["fov_id"] + ".ome.tiff.part")
            if partial.is_file():
                partial.unlink()
        time.sleep(2 ** (attempt + 1))
if RECEIPTS.exists():
    if json.loads(RECEIPTS.read_text()) != downloaded:
        raise ValueError("Retained original download report changed")
else:
    RECEIPTS.write_text(json.dumps(downloaded, indent=2) + "\n")
if not VOLUMES.exists():
    print(prepare_volumes(PLAN, RECEIPTS, RAW, VOLUMES))
metadata, identity, fields = load_volumes(VOLUMES / "volumes.json")
print("Verified development volumes:", len(fields))
print("Target units:", metadata["target_units"])


## Measure GPU time before starting the full run

This benchmark uses synthetic arrays solely to measure time and memory. It
produces **no biological performance result**. Its weights are discarded before
training. The printed training-time estimate excludes download, preprocessing,
validation and checkpoint saving.


In [ ]:
from virtual_staining.reference_trainer import make_model

model = make_model(32, 4, "allen-normal").cuda()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.5, 0.999))
x = torch.randn(BATCH_SIZE, 1, 32, 64, 64, device="cuda")
y = torch.randn_like(x)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
for _ in range(3):
    optimiser.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x), y)
    loss.backward()
    optimiser.step()
torch.cuda.synchronize()
seconds = (time.perf_counter() - started) / 3
print(f"Observed seconds/update: {seconds:.3f}")
print(f"Training-only estimate: {seconds * STEPS / 60:.1f} minutes, excluding validation/data work")
print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
del model, optimiser, x, y, loss
torch.cuda.empty_cache()


In [ ]:
from virtual_staining.reference_trainer import train_reference

torch.set_num_threads(2)
summary = train_reference(
    VOLUMES / "volumes.json", OUTPUT,
    steps=STEPS, width=32, depth=4, patch_shape=(32, 64, 64),
    batch_size=BATCH_SIZE, validate_every=500, seed=17, device="cuda",
    initialisation="allen-normal",
)
print(json.dumps(summary, indent=2))


In [ ]:
from IPython.display import display, Image

report = json.loads((OUTPUT / "training.json").read_text())
print("Scope:", report["scope"])
print("Best complete-volume validation:", report["best_validation"])
print("Per-field results:")
for row in report["best_fields"]:
    print(row)
for image in sorted(OUTPUT.glob("validation-*.png")):
    print(image.name)
    display(Image(filename=str(image)))
print("These are development results, not the untouched test evaluation.")
print("Do not claim useful staining from a low error alone: check nuclear locations, missed nuclei and false structures.")
